In [1]:
import pandas as pd
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict


tokenizer = AutoTokenizer.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture
)

token_length = 256 # adjust to maximal token length

# restricting legth (make room for cls token and paragraph seperators)
TOK_LEN = token_length - 10

/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# read  data
df = pd.read_parquet('/raid/deallab/SF_RAG_Data/ASQA/train.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [3]:
#parse tables to text
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)

    return '\n'.join(table_text)

# parse pars to text
def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4', 'h5'])
    try:
        heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    except:
        print(heading)
        raise
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

#pars unordered lists to text
def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

# pars ordered list to text
def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

# parse whole document
def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4', 'h5'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



In [4]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

#creating question evidence pairs for retrival training (not implemented)
qe_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'evidence_id'])

#fetched_documents = {}
for idx, row in df.iterrows():
    if idx == 100: break # change number of document to chunk/process
    sample_id = row['sample_id']
    evidences = row['wikipages']
    q1 = row['ambiguous_question']
    q2 = defaultdict(list)
    for q in row['qa_pairs']:
        question = q['question']
        wikipage = q['wikipage']
        if not question or not wikipage: continue
        q2[wikipage].append(question)
    for evidence in evidences:
        url = evidence['url']
        #if url in fetched_documents: continue
        #fetched_documents[url] = []
        page = requests.get(url)
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(id='firstHeading').get_text()
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = [[]]
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < TOK_LEN:
                documents[-1].extend(tokenized_par)
            elif length > TOK_LEN:
                begin = 0 
                while begin < length:
                    if begin + TOK_LEN >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + TOK_LEN])
                    begin += TOK_LEN - int(TOK_LEN * 0.1)
            else:
                documents.append(tokenized_par)
            
        print(len(documents))
        for doc in documents:
            doc_text = tokenizer.decode(doc)
            id = uuid4()
            #fetched_documents[url].append(id)
            evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, q1, doc_text]
        
        if title in q2:
            for question in q2[title]:
                for doc in documents:
                    doc_text = tokenizer.decode(doc)
                    id = uuid4()
                    #fetched_documents[url].append(id)
                    evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, question, doc_text]
                
#print(fetched_documents)
evidence_df

319
34
27
27
12
67
107
80
61
75
22
98
259
17
29
13
58
151
259
68
7
12
40
75
3
5
32
53
73
39
52
6
31
13
1
29
107
16
21
24
24
22
32
67
63
104
89
63
17
78
29
71
16
18
112
12
72
10
56
12
21
9
89
11
10
3
14
4
112
40
5
18
8
29
62
15
38
27
3
56
57
31
19
60
28
22
69
23
15
7
32
62
24
97
139
65
55
71
1
22
3
8
9
19
3
76
140
42
4
101
75
20
42
14
445
76
454
363
229
2
44
58
35
132
14
13
1
5
62
25
46
69
40
29
3
74
80
51
11
17
18
28
209
32
11
19
5
8
52
83
120
30
46
5
63
504
7
32
77
9
47


,id,sample_id,title,url,question,text
0,566847d4-7588-4c87-a9a4-ee16abc39bfd,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,bbc92c38-9222-4994-a74a-dcc9740934d3,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,3c7a6a63-30d3-4c71-b7ca-db0f162778ff,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
3,1a081eba-524b-4bd6-b6c7-56d925931b37,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n## Episodes\n\n### Season 1 (2015–16)\n
4,8b35a5bb-40ea-4767-b668-956ba4f88333,-5742327688291876861,List of Bunk'd episodes,https://en.wikipedia.org/wiki/List%20of%20Bunk...,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
...,...,...,...,...,...,...
14083,c91246fc-bf30-41ff-95e6-6d913146f1aa,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,Williams;Nominated;[162]\nBest Original Score...
14084,00ed3f43-237c-4df0-8549-4b1dc744e100,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,2018;Favorite Movie;Star Wars: The Last Jedi;N...
14085,94739b02-c038-4239-80bc-6108dd29b16d,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,;[171]\nDaisy Ridley;Nominated\nChoice Fantasy...
14086,4fa77523-9c7f-462a-a767-4ad935050f8e,7302068382751492271,Star Wars: The Last Jedi,https://en.wikipedia.org/wiki/Star%20Wars%3A%2...,When did Star Wars the Last Jedi come out thro...,"Fujita, Jiyong Shin, and Dan Finnegan for ""Me..."


In [5]:
evidence_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv', index=False)